# Fetch metadata of youtube video

In [ ]:
import pandas as pd 
df = pd.read_csv("Input/Source of de-escalation youtube video playlist.csv")
df.head(5)

In [ ]:
import yt_dlp
import tqdm
import csv
import time
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
    YouTubeRequestFailed
)
# Import DownloadError from yt_dlp for specific error catching
from yt_dlp.utils import DownloadError 

for row in range(1,len(df)-1):
    channel_name = df.loc[row].link.split("@")[1]
    print(f"Processing channel {row}: {channel_name}")

    # ----------------------------
    # 🔧 CONFIGURATION
    # ----------------------------
    # ⚠️ IMPORTANT: Replace this with the URL of the YouTube channel you want to process.
    # Use the channel's main URL or handle URL (e.g., "https://www.youtube.com/@GoogleDevelopers")
    channel_url = f"https://www.youtube.com/@{channel_name}/videos"
    # Alternative fix using the stable Channel ID
    # channel_url = f"https://www.youtube.com/channel/UCe3WJv2YpQ2nJq8oR_910pQ/videos"
    output_file = f"output-video-list/{channel_name}_metadata_and_transcripts.csv"

    # ----------------------------
    # 📺 Extract channel info
    # ----------------------------
    # 'quiet': Suppress standard output
    # 'extract_flat': False ensures we get detailed metadata for all videos
    # NEW: 'ignoreerrors': True ensures the entire channel list generation doesn't crash 
    # when encountering a restricted or deleted video.
    ydl_opts = {'quiet': True, 'extract_flat': False, 'ignoreerrors': True}

    print(f"\nFetching channel metadata...\n")
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            # yt-dlp is smart enough to detect a channel URL and extract its video list
            # By setting 'ignoreerrors': True, this step will proceed even if it hits a restricted video.
            channel_info = ydl.extract_info(channel_url, download=False)
    except Exception as e:
        print(f"❌ Critical Error fetching channel info: {e}")
        print("Please check the channel URL and your network connection. Exiting.")
        # Exit script if initial extraction fails
        channel_info = {}

    # Get channel title and video entries
    channel_title = channel_info.get('title', 'Unknown Channel')
    # Filter out entries that failed extraction (which yt-dlp might leave as None)
    video_entries = [e for e in channel_info.get('entries', []) if e]

    print(f"✅ Channel: {channel_title}")
    print(f"📺 Total videos found (after filtering failed entries): {len(video_entries)}\n")

    # ----------------------------
    # 🧾 Prepare CSV output
    # ----------------------------
    fieldnames = [
        "Index",
        "Title",
        "URL",
        "Uploader",
        "Views",
        "Upload Date",
        "Duration (sec)",
        "Description",
        "Transcript"
    ]
    # Initialize transcript API object
    transcript_fetcher = YouTubeTranscriptApi()

    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        # ----------------------------
        # 🚀 Process each video entry
        # ----------------------------
        for idx, entry in tqdm.tqdm(enumerate(video_entries, start=1)):
            # We need to re-extract the full video info for the specific video 
            # to ensure we get all metadata, as the channel list only provides summary info.
            video_id = entry.get("id")
            video_url = entry.get("webpage_url") or f"https://www.youtube.com/watch?v={video_id}"
            
            # print(f"{idx}. Attempting to fetch details for: {video_url}")

            # --- Use a try/except specifically for fetching the single video details ---
            try:
                # yt-dlp needs to fetch the specific video page for full metadata
                with yt_dlp.YoutubeDL(ydl_opts) as ydl_single:
                    video_details = ydl_single.extract_info(video_url, download=False)

                # Check if video_details contains the expected info, sometimes restricted videos
                # will still be in the list but the extract_info call will fail or return incomplete data.
                if not video_details or video_details.get('_type') == 'url' or not video_details.get('title'):
                    # print(f"   -> ⚠️ Skipped: Video details are incomplete or a non-video item.")
                    continue

                # Extract essential video metadata from the full details
                title = video_details.get("title", "Untitled")
                uploader = video_details.get("uploader", "Unknown")
                views = video_details.get("view_count", "N/A")
                upload_date_raw = video_details.get("upload_date", "N/A")
                duration = video_details.get("duration", 0)
                description = video_details.get("description", "").replace("\n", " ")[:500] + "..."

            except DownloadError as de:
                # Catch the specific yt-dlp download errors (e.g., age-restriction, members-only)
                # print(f"   -> ❌ Skipped: Restricted Content/Download Error for {video_url}. Error: {de}")
                # Write a placeholder entry to the CSV
                writer.writerow({
                    "Index": idx,
                    "Title": f"[RESTRICTED] ID: {video_id}",
                    "URL": video_url,
                    "Uploader": "N/A",
                    "Views": "N/A",
                    "Upload Date": "N/A",
                    "Duration (sec)": "N/A",
                    "Description": "Content is members-only, age-restricted, or private. Metadata retrieval failed.",
                    "Transcript": "[Cannot access restricted content]"
                })
                time.sleep(1)
                continue # Skip to the next video

            except Exception as e:
                # Catch any other unexpected errors during metadata extraction
                # print(f"   -> ❌ Skipped: Unexpected error retrieving metadata for {video_url}. Error: {e}")
                time.sleep(1)
                continue # Skip to the next video


            # ----------------------------
            # 🗣️ Fetch Transcript (Existing logic remains)
            # ----------------------------
            transcript_text = ""
            # try:
            #     # Fetch the transcript, prioritizing English
            #     transcript = transcript_fetcher.fetch(video_id=video_id, languages=["en"])
            #     # Join all transcript snippets into a single string
            #     transcript_text = " ".join(snippet.text for snippet in transcript.snippets)
            # except TranscriptsDisabled:
            #     transcript_text = "[Transcript disabled by uploader]"
            # except NoTranscriptFound:
            #     # Try to fetch automatically generated transcript if primary failed
            #     try:
            #         transcript = transcript_fetcher.fetch(video_id=video_id, languages=["en-auto"])
            #         transcript_text = " ".join(snippet['text'] for snippet in transcript)
            #         # print("   -> Fetched automatically generated transcript.")
            #     except:
            #         transcript_text = "[No transcript available, even automatic]"
            # except YouTubeRequestFailed as e:
            #     transcript_text = f"[Rate limit or request failed: {e}]"
            # except Exception as e:
            #     transcript_text = f"[Error fetching transcript: {e}]"

            # ----------------------------
            # 💾 Write data to CSV
            # ----------------------------
            writer.writerow({
                "Index": idx,
                "Title": title,
                "URL": video_url,
                "Uploader": uploader,
                "Views": views,
                "Upload Date": upload_date_raw,
                "Duration (sec)": duration,
                "Description": description,
                "Transcript": transcript_text
            })

            # Wait a bit to avoid rate limit, especially for the transcript API
            time.sleep(1) 
            
    print(f"\n✅ All done! Metadata + transcripts saved to '{output_file}'")

# Merge all the youtube video in one file

In [ ]:
import os
import pandas as pd

# --- Configuration ---
# !!! IMPORTANT: Update these two paths !!!
input_dir = 'output-video-list/'      # The folder WHERE your CSVs are
output_dir = 'merge-output' # The folder WHERE you want the final file
# ---------------------

output_filename = 'video-list.csv'
all_dataframes = []

# 1. Find and list all CSV files
try:
    # Get all files in the directory and filter for .csv
    csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]
    
    if not csv_files:
        print(f"No CSV files found in '{input_dir}'.")
        exit()
        
    print(f"Found {len(csv_files)} CSV files to merge.")

except FileNotFoundError:
    print(f"Error: Input directory not found at '{input_dir}'")
    exit()
except Exception as e:
    print(f"An error occurred: {e}")
    exit()

# 2. Read each CSV, add new columns, and add to list
for filename in csv_files:
    file_path = os.path.join(input_dir, filename)
    try:
        df = pd.read_csv(file_path)
        
        if df.empty:
            print(f" - Warning: '{filename}' is empty and will be skipped.")
            continue

        # --- MODIFICATIONS (Per-file) ---
        # 1. Add channel_name (first part of filename before '_')
        channel_name = filename.split('_')[0]
        df['channel_name'] = channel_name
        
        # 2. Add 'id' (the index from the original input file - this will be dropped later)
        df['id'] = df.index
        # --- END MODIFICATIONS ---
        
        all_dataframes.append(df)
        print(f" - Read and processed '{filename}'")

    except pd.errors.EmptyDataError:
        print(f" - Warning: '{filename}' is empty and will be skipped.")
    except Exception as e:
        print(f" - Error reading '{filename}': {e}. Skipping this file.")

# 3. Merge (concatenate) all the data into one DataFrame
if not all_dataframes:
    print("No data was successfully read. Exiting.")
else:
    # ignore_index=True is important: it creates a new 0-based index
    # for the *merged* DataFrame.
    merged_df = pd.concat(all_dataframes, ignore_index=True)
    print("\nAll files merged successfully.")

    # --- FINAL MODIFICATIONS ---
    
    # 4. Drop the temporary 'id' column (the "previous index" from input files)
    if 'id' in merged_df.columns:
        merged_df = merged_df.drop(columns=['id'])
        print("Dropped temporary 'id' column (per-file index) from final file.")
    
    # 3. Create the global index column and name it 'id'
    # This is the new, globally incrementing index as requested.
    merged_df['id'] = merged_df.index
    
    # 5. Reorder the columns to make 'id' the first column
    cols = list(merged_df.columns)
    # Remove 'id' from its current (last) position
    cols.remove('id')
    # Insert 'id' at the beginning of the list
    cols.insert(0, 'id')
    
    # Apply the new column order
    merged_df = merged_df[cols]
    
    print("New global index renamed to 'id' and set as the first column.")
    # --- END FINAL MODIFICATIONS ---

    # 6. Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # 7. Save the final merged file
    output_path = os.path.join(output_dir, output_filename)
    
    try:
        # index=False stops pandas from writing an extra index column
        merged_df.to_csv(output_path, index=False)
        print(f"\n✅ Success! Merged file saved to:\n{output_path}")
    except Exception as e:
        print(f"\n❌ Error saving merged file: {e}")

# Update the transcript

In [ ]:
from youtube_transcript_api.proxies import WebshareProxyConfig

In [ ]:
# Fetch youtube transcribe from youtube video and merge it to the data
import tqdm
import time
import pandas as pd
import yt_dlp
import csv
import time
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    TranscriptsDisabled,
    NoTranscriptFound,
    YouTubeRequestFailed
)

# # Initialize transcript API object
# transcript_fetcher = YouTubeTranscriptApi()
# Initialize transcript API object
transcript_fetcher = YouTubeTranscriptApi(
        proxy_config=WebshareProxyConfig(
        proxy_username="jfxywsaa",
        proxy_password="20cip95e4lx7",
    )
)

input_dir = 'merge-output/'      # The folder WHERE your CSVs are
output_dir = 'merge-output/' # The folder WHERE you want the final file
df = pd.read_csv("merge-output/video-list-with-transcripts.csv")

In [ ]:
for i in tqdm.tqdm(range(init_idx, len(df))):
    video_url = df.loc[i].URL
    video_id = video_url.split("v=")[-1]
    # print(video_id)
    transcript_text = ""
    try:
        # Fetch the transcript, prioritizing English
        # time.sleep(1)
        transcript = transcript_fetcher.fetch(video_id=video_id, languages=["en"])
        # print(transcript)
        # Join all transcript snippets into a single string
        transcript_text = " ".join(snippet.text for snippet in transcript.snippets)
    except TranscriptsDisabled:
        print("[Transcript disabled by uploader]")
        df.to_csv("merge-output/video-list-with-transcripts.csv", index=False)
        init_idx = i 
        # i = i + 1
        # break
    except NoTranscriptFound:
        # Try to fetch automatically generated transcript if primary failed
        try:
            transcript = transcript_fetcher.fetch(video_id=video_id, languages=["en-auto"])
            transcript_text = " ".join(snippet.text for snippet in transcript.snippets)
            print("      -> Fetched automatically generated transcript.")
        except Exception:
            print("[No transcript available, even automatic]")
            df.to_csv("merge-output/video-list-with-transcripts.csv", index=False)
            init_idx = i
            # i = i + 1
            # break
    except YouTubeRequestFailed as e:
        transcript_text = f"[Rate limit or request failed: {e}]"
        df.to_csv("merge-output/video-list-with-transcripts.csv", index=False)
        init_idx = i
        # break
    except Exception as e:
        print(f"[Error fetching transcript: {e}]")
        df.to_csv("merge-output/video-list-with-transcripts.csv", index=False)
        init_idx = i
        # break
    # print(transcript_text)
    df.loc[i, 'Transcript'] = transcript_text
    
df.to_csv("merge-output/video-list-with-transcripts.csv", index=False)

In [ ]:
init_idx